# Imports and Shared Configuration


In [ ]:
# Install the YOLO dependency and import all libraries used in the notebook.

!pip install -q ultralytics

import os
import random
import shutil
import xml.etree.ElementTree as ET

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from ultralytics import YOLO

DATASET_PATH = "/kaggle/input/datasets/alimaged10/pcb-dataset/PCB-DATASET-master"
LABELS_DIR = "/kaggle/working/pcb_yolo_labels"
YOLO_ROOT = "/kaggle/working/PCB_YOLO"
BASELINE_PROJECT = "/kaggle/working/PCB_Baseline"
BASELINE_RUN_NAME = "YOLO11s_800"

CLASSES = [
    "Missing_hole",
    "Mouse_bite",
    "Open_circuit",
    "Short",
    "Spur",
    "Spurious_copper"
]

CLASS_TO_ID = {
    cls: idx
    for idx, cls in enumerate(CLASSES)
}

CLASS_NAMES = {
    idx: cls
    for idx, cls in enumerate(CLASSES)
}

# Lowercase aliases keep the existing notebook cells readable without repeating declarations.
dataset_path = DATASET_PATH
labels_dir = LABELS_DIR
yolo_root = YOLO_ROOT
classes = CLASSES
class_to_id = CLASS_TO_ID
class_names = CLASS_NAMES


# Reproducibility


In [ ]:
# Set fixed random seeds for reproducible splitting, sampling, and training behavior.

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"Seed set to {SEED}")


# Exploratory Data Analysis


In [ ]:
# Preview the PCB dataset folder structure.

for root, dirs, files in os.walk(dataset_path):
    level = root.replace(dataset_path, '').count(os.sep)
    indent = ' ' * 4 * level
    print(f"{indent}{os.path.basename(root)}/")
    
    if level < 2:  # prevent huge output
        subindent = ' ' * 4 * (level + 1)
        for f in files[:10]:
            print(f"{subindent}{f}")

# Baseline: YOLO Dataset Preparation


## Step 1: Convert XML Annotations to YOLO Label Files


In [ ]:
# Convert XML annotations into YOLO-format label text files.

os.makedirs(labels_dir, exist_ok=True)

# ==========================================================
# Conversion
# ==========================================================

num_images = 0
num_boxes = 0

for cls in classes:

    ann_dir = os.path.join(
        dataset_path,
        "Annotations",
        cls
    )

    img_dir = os.path.join(
        dataset_path,
        "images",
        cls
    )

    class_id = class_to_id[cls]

    for xml_file in os.listdir(ann_dir):

        if not xml_file.endswith(".xml"):
            continue

        xml_path = os.path.join(
            ann_dir,
            xml_file
        )

        image_name = xml_file.replace(
            ".xml",
            ".jpg"
        )

        image_path = os.path.join(
            img_dir,
            image_name
        )

        if not os.path.exists(image_path):
            continue

        # Read image dimensions
        img = cv2.imread(image_path)

        if img is None:
            continue

        h, w = img.shape[:2]

        tree = ET.parse(xml_path)
        root = tree.getroot()

        yolo_lines = []

        for obj in root.findall("object"):

            bbox = obj.find("bndbox")

            xmin = float(bbox.find("xmin").text)
            ymin = float(bbox.find("ymin").text)
            xmax = float(bbox.find("xmax").text)
            ymax = float(bbox.find("ymax").text)

            # YOLO format
            x_center = ((xmin + xmax) / 2) / w
            y_center = ((ymin + ymax) / 2) / h

            box_w = (xmax - xmin) / w
            box_h = (ymax - ymin) / h

            yolo_lines.append(
                f"{class_id} "
                f"{x_center:.6f} "
                f"{y_center:.6f} "
                f"{box_w:.6f} "
                f"{box_h:.6f}"
            )

            num_boxes += 1

        label_path = os.path.join(
            labels_dir,
            image_name.replace(".jpg", ".txt")
        )

        with open(label_path, "w") as f:
            f.write("\n".join(yolo_lines))

        num_images += 1

print(f"Converted Images : {num_images}")
print(f"Converted Boxes  : {num_boxes}")
print(f"Labels Saved To  : {labels_dir}")

In [ ]:
# Inspect the generated YOLO label files and print one sample label file.

label_files = [
    f for f in os.listdir(labels_dir)
    if f.endswith(".txt")
]

print("Label Files:", len(label_files))

sample = sorted(label_files)[0]

print("\nSample Label File:")
print(sample)

with open(os.path.join(labels_dir, sample)) as f:
    print(f.read())

In [ ]:
# Count the total number of YOLO boxes written across all label files.

total_boxes = 0

for file in os.listdir(labels_dir):

    if file.endswith(".txt"):

        with open(os.path.join(labels_dir, file)) as f:
            total_boxes += len(f.readlines())

print("Total Boxes:", total_boxes)

In [ ]:
# Check whether image file names are unique across all class folders.

all_names = []

for cls in classes:

    img_dir = os.path.join(dataset_path, "images", cls)

    for img_name in os.listdir(img_dir):

        if img_name.endswith(".jpg"):
            all_names.append(img_name)

print("Total images:", len(all_names))
print("Unique names:", len(set(all_names)))

## Step 2: Split Images and Build the YOLO Folder Structure


In [ ]:
# Build a DataFrame listing all images, classes, and source paths.

records = []

for cls in classes:

    img_dir = os.path.join(
        dataset_path,
        "images",
        cls
    )

    for img_name in os.listdir(img_dir):

        if not img_name.lower().endswith(".jpg"):
            continue

        records.append([
            img_name,
            cls,
            os.path.join(img_dir, img_name)
        ])

df = pd.DataFrame(
    records,
    columns=[
        "image_name",
        "class",
        "image_path"
    ]
)

print("Total Images:", len(df))
df.head()

In [ ]:
# Split the image DataFrame into train, validation, and test sets.

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=42,
    shuffle=True
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=1/3,
    random_state=42,
    shuffle=True
)

print("Train:", len(train_df))
print("Val  :", len(val_df))
print("Test :", len(test_df))
print("Total:", len(train_df)+len(val_df)+len(test_df))

In [ ]:
# Create the YOLO directory structure for images and labels.

folders = [

    "images/train",
    "images/val",
    "images/test",

    "labels/train",
    "labels/val",
    "labels/test"
]

for folder in folders:

    os.makedirs(
        os.path.join(yolo_root, folder),
        exist_ok=True
    )

print("Folders created.")

In [ ]:
# Copy images and label files into their train, validation, and test folders.

splits = {
    "train": train_df,
    "val": val_df,
    "test": test_df
}

for split_name, split_df in splits.items():

    for _, row in split_df.iterrows():

        image_name = row["image_name"]

        image_src = row["image_path"]

        label_src = os.path.join(
            labels_dir,
            image_name.replace(".jpg", ".txt")
        )

        image_dst = os.path.join(
            yolo_root,
            "images",
            split_name,
            image_name
        )

        label_dst = os.path.join(
            yolo_root,
            "labels",
            split_name,
            image_name.replace(".jpg", ".txt")
        )

        shutil.copy2(
            image_src,
            image_dst
        )

        shutil.copy2(
            label_src,
            label_dst
        )

print("Dataset copied.")

In [ ]:
# Verify that each split has the expected number of images and labels.

for split in ["train","val","test"]:

    n_images = len(
        os.listdir(
            os.path.join(
                yolo_root,
                "images",
                split
            )
        )
    )

    n_labels = len(
        os.listdir(
            os.path.join(
                yolo_root,
                "labels",
                split
            )
        )
    )

    print(
        f"{split:<5} "
        f"Images={n_images} "
        f"Labels={n_labels}"
    )

In [ ]:
# Write the YOLO data.yaml configuration file.

yaml_content = """
path: /kaggle/working/PCB_YOLO

train: images/train
val: images/val
test: images/test

names:
  0: Missing_hole
  1: Mouse_bite
  2: Open_circuit
  3: Short
  4: Spur
  5: Spurious_copper
"""

with open(
    os.path.join(yolo_root, "data.yaml"),
    "w"
) as f:

    f.write(yaml_content)

print("data.yaml created")

In [ ]:
# Visualize random training images with YOLO labels converted back to boxes.

image_dir = os.path.join(
    yolo_root,
    "images",
    "train"
)

label_dir = os.path.join(
    yolo_root,
    "labels",
    "train"
)

samples = random.sample(
    os.listdir(image_dir),
    6
)

fig, axes = plt.subplots(
    2,
    3,
    figsize=(18,10)
)

for ax, image_name in zip(
    axes.ravel(),
    samples
):

    image_path = os.path.join(
        image_dir,
        image_name
    )

    label_path = os.path.join(
        label_dir,
        image_name.replace(".jpg", ".txt")
    )

    img = cv2.imread(image_path)
    img = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2RGB
    )

    h, w = img.shape[:2]

    with open(label_path) as f:

        lines = f.readlines()

    for line in lines:

        cls, xc, yc, bw, bh = map(
            float,
            line.strip().split()
        )

        x1 = int((xc - bw/2) * w)
        y1 = int((yc - bh/2) * h)

        x2 = int((xc + bw/2) * w)
        y2 = int((yc + bh/2) * h)

        cv2.rectangle(
            img,
            (x1, y1),
            (x2, y2),
            (255,0,0),
            3
        )

        cv2.putText(
            img,
            class_names[int(cls)],
            (x1, y1-5),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (255,0,0),
            2
        )

    ax.imshow(img)
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Visualize one labeled training example from each defect class.

image_dir = os.path.join(yolo_root, "images", "train")
label_dir = os.path.join(yolo_root, "labels", "train")

selected = {}

for img_name in os.listdir(image_dir):

    label_path = os.path.join(
        label_dir,
        img_name.replace(".jpg", ".txt")
    )

    with open(label_path) as f:

        lines = f.readlines()

    for line in lines:

        cls = int(line.split()[0])

        if cls not in selected:

            selected[cls] = img_name

        if len(selected) == 6:
            break

    if len(selected) == 6:
        break

fig, axes = plt.subplots(
    2,
    3,
    figsize=(18,10)
)

for ax, cls in zip(
    axes.ravel(),
    sorted(selected.keys())
):

    img_name = selected[cls]

    img = cv2.imread(
        os.path.join(image_dir, img_name)
    )

    img = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2RGB
    )

    h, w = img.shape[:2]

    with open(
        os.path.join(
            label_dir,
            img_name.replace(".jpg", ".txt")
        )
    ) as f:

        lines = f.readlines()

    for line in lines:

        c, xc, yc, bw, bh = map(
            float,
            line.split()
        )

        x1 = int((xc - bw/2)*w)
        y1 = int((yc - bh/2)*h)
        x2 = int((xc + bw/2)*w)
        y2 = int((yc + bh/2)*h)

        cv2.rectangle(
            img,
            (x1,y1),
            (x2,y2),
            (255,0,0),
            3
        )

    ax.imshow(img)
    ax.set_title(class_names[cls])
    ax.axis("off")

plt.tight_layout()
plt.show()

## Baseline Training: Train, Validate, and Visualize YOLO11s


In [ ]:
# Check the Torch version, CUDA availability, and GPU name.

print("Torch Version :", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# Print additional Torch and CUDA runtime details.

print(torch.__version__)
print(torch.version.cuda)

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
# Verify the YOLO dataset folders before training.

dataset_root = yolo_root

for split in ["train", "val", "test"]:

    image_dir = os.path.join(
        dataset_root,
        "images",
        split
    )

    label_dir = os.path.join(
        dataset_root,
        "labels",
        split
    )

    print(
        f"{split.upper():<6}",
        "Images:",
        len(os.listdir(image_dir)),
        "| Labels:",
        len(os.listdir(label_dir))
    )

In [ ]:
# Load the YOLO11s model architecture and print its structure.

model = YOLO("yolo11s.pt")

print(model.model)

In [ ]:
# Run a tiny CUDA matrix multiplication sanity check.

x = torch.randn(10,10).cuda()
y = torch.randn(10,10).cuda()

print((x @ y).shape)

In [ ]:
# Run a one-epoch smoke test to confirm YOLO training works.

model = YOLO("yolo11s.pt")

model.train(
    data=os.path.join(yolo_root, "data.yaml"),
    epochs=1,
    imgsz=640,
    batch=4
)

In [ ]:
# Load the smoke-test best model and generate validation predictions.

best_model = YOLO(
    "/kaggle/working/runs/detect/train/weights/best.pt"
)

results = best_model.predict(
    source=os.path.join(yolo_root, "images", "val"),
    conf=0.01,
    save=False
)

print("Predictions generated:", len(results))

In [ ]:
# Display a grid of validation prediction visualizations.

results_sample = random.sample(results, 9)

fig, axes = plt.subplots(3,3, figsize=(15,15))

for ax, r in zip(axes.ravel(), results_sample):

    img = r.plot()

    ax.imshow(img[..., ::-1])
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Train the YOLO11s baseline for the full configured run.

model = YOLO("yolo11s.pt")

results = model.train(
    data=os.path.join(yolo_root, "data.yaml"),

    epochs=100,

    imgsz=800,

    batch=8,

    workers=2,

    seed=42,

    patience=20,

    fliplr=0.5,
    flipud=0.5,

    degrees=20,

    translate=0.1,
    scale=0.2,

    mosaic=1.0,
    mixup=0,

    project=BASELINE_PROJECT,
    name=BASELINE_RUN_NAME,
    exist_ok=True
)

In [ ]:
#       Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
#      98/100       5.3G      1.145     0.5052     0.8958         20        800: 100% ━━━━━━━━━━━━ 61/61 3.2it/s 18.9s0.2s
#                  Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 5.1it/s 1.8s0.2s
#                    all        138        586      0.966      0.951      0.967      0.519

#       Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
#      99/100       5.3G      1.151     0.5055     0.8962         18        800: 100% ━━━━━━━━━━━━ 61/61 3.1it/s 19.4s0.2s
#                  Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 4.9it/s 1.8s0.2s
#                    all        138        586      0.968      0.946      0.968      0.517

#       Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
#     100/100       5.3G      1.122     0.4968     0.8925         20        800: 100% ━━━━━━━━━━━━ 61/61 3.4it/s 18.1s0.2s
#                  Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 4.3it/s 2.1s0.3s
#                    all        138        586      0.966       0.95      0.967      0.522

# 100 epochs completed in 0.619 hours.
# Optimizer stripped from /kaggle/working/PCB_Baseline/YOLO11s_800/weights/last.pt, 19.2MB
# Optimizer stripped from /kaggle/working/PCB_Baseline/YOLO11s_800/weights/best.pt, 19.2MB

# Validating /kaggle/working/PCB_Baseline/YOLO11s_800/weights/best.pt...
# Ultralytics 8.4.69 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
# YOLO11s summary (fused): 101 layers, 9,415,122 parameters, 0 gradients, 21.3 GFLOPs
#                  Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 3.3it/s 2.7s0.2s
#                    all        138        586      0.969      0.947      0.967      0.529
#           Missing_hole         31        128      0.995      0.984      0.985      0.569
#             Mouse_bite         26        115      0.968      0.904      0.955      0.516
#           Open_circuit         21         85      0.987      0.917      0.979      0.546
#                  Short         18         81      0.976      0.999       0.99      0.556
#                   Spur         18         77      0.942      0.909       0.92       0.47
#        Spurious_copper         24        100      0.947       0.97       0.97      0.517
# Speed: 0.3ms preprocess, 7.3ms inference, 0.1ms loss, 3.8ms postprocess per image
# Results saved to /kaggle/working/PCB_Baseline/YOLO11s_800

In [ ]:
# Compare ground-truth boxes and model predictions on random test images.

# ==========================================================
# Load model
# ==========================================================

model = YOLO(
    os.path.join(BASELINE_PROJECT, BASELINE_RUN_NAME, "weights", "best.pt")
)

# ==========================================================
# Paths
# ==========================================================

image_dir = os.path.join(yolo_root, "images", "test")
label_dir = os.path.join(yolo_root, "labels", "test")

# ==========================================================
# Random images
# ==========================================================

image_files = [
    f for f in os.listdir(image_dir)
    if f.endswith(".jpg")
]

sample_images = random.sample(image_files, 20)

# ==========================================================
# Plot
# ==========================================================

fig, axes = plt.subplots(
    20,
    2,
    figsize=(16, 100)
)

for row, img_name in enumerate(sample_images):

    image_path = os.path.join(
        image_dir,
        img_name
    )

    label_path = os.path.join(
        label_dir,
        img_name.replace(".jpg", ".txt")
    )

    # ------------------------------------------------------
    # Original image
    # ------------------------------------------------------

    img = cv2.imread(image_path)
    img = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2RGB
    )

    h, w = img.shape[:2]

    # ======================================================
    # GROUND TRUTH IMAGE
    # ======================================================

    gt_img = img.copy()

    with open(label_path) as f:

        lines = f.readlines()

    for line in lines:

        cls, xc, yc, bw, bh = map(
            float,
            line.strip().split()
        )

        cls = int(cls)

        x1 = int((xc - bw/2) * w)
        y1 = int((yc - bh/2) * h)
        x2 = int((xc + bw/2) * w)
        y2 = int((yc + bh/2) * h)

        cv2.rectangle(
            gt_img,
            (x1, y1),
            (x2, y2),
            (0,255,0),
            3
        )

        cv2.putText(
            gt_img,
            class_names[cls],
            (x1, max(25, y1-5)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0,255,0),
            2
        )

    # ======================================================
    # PREDICTION IMAGE
    # ======================================================

    pred_img = img.copy()

    result = model.predict(
        image_path,
        conf=0.25,
        verbose=False
    )[0]

    for box in result.boxes:

        cls = int(box.cls[0])
        conf = float(box.conf[0])

        x1, y1, x2, y2 = map(
            int,
            box.xyxy[0]
        )

        cv2.rectangle(
            pred_img,
            (x1, y1),
            (x2, y2),
            (255,0,0),
            3
        )

        cv2.putText(
            pred_img,
            f"{class_names[cls]} {conf:.2f}",
            (x1, max(25, y1-5)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (255,0,0),
            2
        )

    # ======================================================
    # Display
    # ======================================================

    axes[row,0].imshow(pred_img)
    axes[row,0].set_title(
        f"Prediction - {img_name}"
    )
    axes[row,0].axis("off")

    axes[row,1].imshow(gt_img)
    axes[row,1].set_title(
        f"Ground Truth - {img_name}"
    )
    axes[row,1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Display the saved YOLO training results plot.

img = cv2.imread(
    os.path.join(BASELINE_PROJECT, BASELINE_RUN_NAME, "results.png")
)

img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(14,10))
plt.imshow(img)
plt.axis("off")
plt.show()